# 07 Active Learning for Data Expansion

This notebook provides an interactive interface for performing Active Learning. It loads the trained baseline model, calculates prediction uncertainty (using Entropy, Margin, or Least Confidence) over a pool of unlabeled Facebook comments, visualizes the uncertainty distribution, and exports the most uncertain comments for human annotation.

In [1]:
import sys
from pathlib import Path

# Ensure project root is in sys.path
PROJECT_ROOT = Path.cwd().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from src.data_utils import build_paths
from src.active_learning import run_active_learning

paths = build_paths(PROJECT_ROOT)
print("Paths loaded:")
print(f"Models dir: {paths.models_dir}")
print(f"Raw data dir: {paths.raw_dir}")

Paths loaded:
Models dir: C:\Users\vdchi\OneDrive\Documents\DATA SCIENCE\Sem5\Group Project\Prototyping 2\models
Raw data dir: C:\Users\vdchi\OneDrive\Documents\DATA SCIENCE\Sem5\Group Project\Prototyping 2\data\raw


## 1. Specify Unlabeled Pool Path

Specify the path to the CSV file containing scraped, unlabeled Facebook comments. 
*For testing, we can use the mock dataset under `data/scratch/mock_unlabeled.csv`.*

In [2]:
# Update this to your scraped, unlabeled CSV pool
UNLABELED_POOL_PATH = paths.root / 'data' / 'scratch' / 'annotator_1_data.csv'

# Read a few rows of the pool
pool_df = pd.read_csv(UNLABELED_POOL_PATH)
print(f"Total unlabeled comments in pool: {len(pool_df)}")
pool_df.head()

Total unlabeled comments in pool: 69


,Unnamed: 0,text
0,0,Amalawi ndife anthu opusa kwambiri. We get eas...
1,1,"When you're single, you see happy couples ever..."
2,2,Where are you Dr Laz 🔥🔥
3,3,"Clear example of talk is cheap,when in opposit..."
4,4,At least that Pastor can now sleep in peace. “...


## 2. Run Active Learning Selection

We will compute the prediction uncertainty for all comments in the pool using the best-performing baseline model.

In [3]:
# Run active learning extraction
STRATEGY = 'entropy'  # Choices: 'entropy', 'margin', 'lc'
OUTPUT_BATCH_PATH = paths.raw_dir / 'Facebook_Comment_Annotation - ActiveBatch1.csv'
N_SAMPLES = 10

candidates = run_active_learning(
    unlabeled_path=UNLABELED_POOL_PATH,
    output_path=OUTPUT_BATCH_PATH,
    n_samples=N_SAMPLES,
    strategy=STRATEGY,
    root=PROJECT_ROOT
)

2026-05-27 06:37:12,579 - INFO - Loading baseline model and vectorizer...
2026-05-27 06:37:13,959 - INFO - Loading unlabeled data from C:\Users\vdchi\OneDrive\Documents\DATA SCIENCE\Sem5\Group Project\Prototyping 2\data\scratch\annotator_1_data.csv...
2026-05-27 06:37:13,969 - INFO - Generating 'id' column for unlabeled comments...
2026-05-27 06:37:13,975 - INFO - Cleaning text and extracting features...
2026-05-27 06:37:14,050 - INFO - Computing uncertainty using strategy: entropy...
2026-05-27 06:37:14,055 - INFO - Comparing with existing annotations to prevent duplicates...
2026-05-27 06:37:14,227 - INFO - Dropped 0 comments that are already annotated.
2026-05-27 06:37:14,301 - INFO - Successfully exported 10 uncertainty-selected comments to: C:\Users\vdchi\OneDrive\Documents\DATA SCIENCE\Sem5\Group Project\Prototyping 2\data\raw\Facebook_Comment_Annotation - ActiveBatch1.csv


## 3. Visualize Uncertainty Distribution

Visualizing the uncertainty scores helps us see whether most comments are easy (low uncertainty) or hard (high uncertainty) for the model. High uncertainty indicates comments that are close to the classification boundaries.

In [ ]:
plt.figure(figsize=(8, 5))
sns.histplot(candidates['uncertainty_score'], kde=True, bins=10, color='purple')
plt.title(f'Active Learning Candidate Uncertainty Distribution ({STRATEGY.capitalize()})')
plt.xlabel('Uncertainty Score')
plt.ylabel('Frequency')
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

## 4. Inspect Top Uncertain Candidates

Let's review the comments that the model is most unsure about. These will be written to the export CSV file.

In [ ]:
print(f"--- TOP {N_SAMPLES} MOST UNCERTAIN COMMENTS ---")
for idx, row in candidates.head(N_SAMPLES).iterrows():
    print(f"\nID: {row['id']} | Score: {row['uncertainty_score']:.4f}")
    print(f"Text: {row['text']}")